# 01 — Data Ingestion & Validation
> Downloads the EPA dC/dN dataset, inspects raw files, and runs schema validation.

**Run this notebook first.** All subsequent notebooks depend on the files saved in `data/raw/`.

## 0 · Google Colab Setup
_Skip if running locally — this cell auto-installs deps and mounts the repo._

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/tree_carbon_ml.git 2>/dev/null || true
    %cd tree_carbon_ml
    %pip install -r requirements.txt -q
else:
    # If running locally inside the 'notebooks' folder, move up to the project root
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
# Add src to path
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f'Environment ready. Working directory is now: {os.getcwd()}')

## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.download import DataDownloader
from data.validate import DataValidator, EXPECTED_COLUMNS

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

## 2 · Download Raw Data
Files are downloaded from the EPA ScienceHub and saved to `data/raw/`.

In [ ]:
downloader = DataDownloader(raw_dir='data/raw')
paths = downloader.download_all(force=False)
print('\nPaths:', {k: str(v) for k, v in paths.items()})

## 3 · Load & Inspect Raw Files

In [ ]:
# Load column key (metadata)
col_key = pd.read_csv(paths['column_key'])
print(f'Column key shape: {col_key.shape}')
col_key

In [ ]:
# Load main dataset
df_raw = pd.read_csv(paths['main'])
print(f'Main dataset shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {df_raw.columns.tolist()}')

In [ ]:
# First 5 rows
df_raw.head()

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Basic statistics
df_raw.describe(include='all').T

## 4 · Schema Validation

In [ ]:
validator = DataValidator(df_raw)
validator.run_all().report()

## 5 · Missing Value Overview

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)

summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Columns with any missing values:')
display(summary[summary.missing_count > 0])
print(f'\nTotal rows: {len(df_raw):,}')

## 6 · Duplicate Check

In [ ]:
n_dupes = df_raw.duplicated(subset=['PLT_CN']).sum()
print(f'Duplicate PLT_CN rows: {n_dupes}')
print(f'Unique plots: {df_raw["PLT_CN"].nunique():,}')

## 7 · Geographic Coverage

In [ ]:
print('Latitude  range:', df_raw['LAT'].min(), '→', df_raw['LAT'].max())
print('Longitude range:', df_raw['LON'].min(), '→', df_raw['LON'].max())
print('States covered :', df_raw['STATECD'].nunique())
print('Counties covered:', df_raw['COUNTYCD'].nunique())

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(df_raw['LON'], df_raw['LAT'], s=1, alpha=0.3, c='steelblue')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'FIA Plot Locations (n={len(df_raw):,})')
plt.tight_layout(); plt.show()

## 8 · Ecoregion Coverage

In [ ]:
for col in ['NA_L1CODE','NA_L3CODE','US_L4CODE']:
    print(f'{col}: {df_raw[col].nunique()} unique values')

## 9 · Save Validated Raw Copy
Save a clean copy (renamed columns, parsed types) as the canonical input for downstream notebooks.

In [ ]:
!pip install pyarrow

In [ ]:
import os
os.makedirs('data/interim', exist_ok=True)

# Cast PLT_CN to string for safe joins
df_raw['PLT_CN'] = df_raw['PLT_CN'].astype(str)

out_path = 'data/interim/01_raw_validated.parquet'
df_raw.to_parquet(out_path, index=False)
print(f'Saved to {out_path}  shape={df_raw.shape}')

---
### ✅ Notebook 01 Complete
Next: **02_eda_exploration.ipynb**